<a href="https://colab.research.google.com/github/jagadeeshdandu/NASSCOM-AI-FDP/blob/main/NASSCOM_Day_3_Data_Cleaning_Part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)


Setup complete. pandas 2.2.2


In [2]:
# -----------------------------------------------------------
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# -----------------------------------------------------------
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


In [3]:
# -----------------------------------------------------------
# 🔹 1A. A FEW COMMANDS REVEAL MOST PROBLEMS
# -----------------------------------------------------------

df = raw.copy()        # always work on a copy
df.info()              # types + non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [4]:
# -----------------------------------------------------------
# 🔹 1B. MISSING COUNTS, DUPLICATES & RANGES
# -----------------------------------------------------------

print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [5]:
# duplicate row count
print('Duplicate rows:', df.duplicated().sum())

Duplicate rows: 1


In [6]:
print('Missing per column:')
print(df.isna().sum())

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [7]:
# -----------------------------------------------------------
# 🔹 2A. UNMASK DISGUISED MISSING VALUES
# -----------------------------------------------------------

# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())

Missing after unmasking:
age      2
spend    1
dtype: int64


In [8]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE THE GAPS (impute)
# -----------------------------------------------------------

# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

Missing after imputing: 0


In [9]:
# unmask missing values (spend -> numeric, age -1 -> NaN)

display(df.head(5))

,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,120.5,2024-01-08
4,5,Eve,29.5,Mumbai,150.0,2024-01-09


In [10]:
# Drop rows with any NaN values
df_dropna = df.dropna()
print("DataFrame after dropping rows with NaN values:")
display(df_dropna.head(5))

DataFrame after dropping rows with NaN values:


,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,120.5,2024-01-08
4,5,Eve,29.5,Mumbai,150.0,2024-01-09


In [12]:
print("Median-imputed version of the DataFrame (first 5 rows):")
display(df.head(5))

Median-imputed version of the DataFrame (first 5 rows):


,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,120.5,2024-01-08
4,5,Eve,29.5,Mumbai,150.0,2024-01-09


In [13]:
# compare row counts

print(f"Row count of original 'raw' DataFrame: {raw.shape[0]}")
print(f"Row count of median-imputed 'df' DataFrame: {df.shape[0]}")
print(f"Row count of 'df_dropna' DataFrame (after dropping NaNs): {df_dropna.shape[0]}")

Row count of original 'raw' DataFrame: 8
Row count of median-imputed 'df' DataFrame: 8
Row count of 'df_dropna' DataFrame (after dropping NaNs): 8


In [14]:
# -----------------------------------------------------------
# 🔹 3A. DROP DUPLICATE ROWS
# -----------------------------------------------------------

print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


In [15]:
# -----------------------------------------------------------
# 🔹 3B. FIX DATA TYPES
# -----------------------------------------------------------

# 'date' is text -> convert to real datetimes so sorting/maths work
df['date'] = pd.to_datetime(df['date'])
# 'city' is a category -> mark it as such (saves memory, signals intent)
df['city'] = df['city'].astype('string')
print(df.dtypes)

id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


/tmp/ipykernel_5297/2479632645.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/tmp/ipykernel_5297/2479632645.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['city'] = df['city'].astype('string')


In [20]:
ex = raw.copy()

In [21]:
# Re-comparing row counts after dropping duplicates
print(f"Row count of original 'raw' DataFrame: {raw.shape[0]}")
print(f"Row count of median-imputed and de-duplicated 'df' DataFrame: {df.shape[0]}")
print(f"Row count of 'df_dropna' DataFrame (before de-duplication): {df_dropna.shape[0]}")


Row count of original 'raw' DataFrame: 8
Row count of median-imputed and de-duplicated 'df' DataFrame: 7
Row count of 'df_dropna' DataFrame (before de-duplication): 8


In [22]:
# fix types: spend -> numeric, date -> datetime

print(df.dtypes)

id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [23]:
df.describe()

,id,age,spend,date
count,7.000000,7.000000,7.000000,7
mean,4.000000,31.714286,14254.457143,2024-01-08 00:00:00
min,1.000000,25.000000,80.000000,2024-01-05 00:00:00
25%,2.500000,29.250000,115.250000,2024-01-06 12:00:00
50%,4.000000,29.500000,120.500000,2024-01-08 00:00:00
75%,5.500000,34.000000,175.100000,2024-01-09 12:00:00
max,7.000000,41.000000,99000.000000,2024-01-11 00:00:00
std,2.160247,5.641049,37369.290604,NaN


In [26]:
# dtypes + shape

print(df.dtypes)

id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [27]:
print(df.shape)

(7, 6)


In [28]:
# -----------------------------------------------------------
# 🔹 4A. THE IQR RULE
# -----------------------------------------------------------

# spend has a 99000 value among ~100s -> a clear outlier
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0


In [29]:
# -----------------------------------------------------------
# 🔹 4B. ONE WAY TO TREAT THEM — CAP (winsorise)
# -----------------------------------------------------------

# clip values to the IQR bounds instead of deleting the row
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


### Removing the Outlier Row

Since capping/clipping was not desired, I will now remove the row that contains the identified 'spend' outlier. This is another common way to handle extreme values that might disproportionately influence analysis.

In [31]:
# Q1, Q3, IQR for 'age'

q1_age, q3_age = df['age'].quantile([0.25, 0.75])
iqr_age = q3_age - q1_age
print(f'Age Q1={q1_age:.1f}  Q3={q3_age:.1f}  IQR={iqr_age:.1f}')

Age Q1=29.2  Q3=34.0  IQR=4.8


In [35]:
# lower & upper bounds

low_age, high_age = q1_age - 1.5 * iqr_age, q3_age + 1.5 * iqr_age
print(f'Normal range for Age: {low_age:.1f} to {high_age:.1f}')

Normal range for Age: 22.1 to 41.1


In [34]:
# rows outside the bounds

outliers_age = df[(df['age'] < low_age) | (df['age'] > high_age)]
print('\nOutlier rows based on Age:')
print(outliers_age[['name', 'age']])


Outlier rows based on Age:
Empty DataFrame
Columns: [name, age]
Index: []


In [36]:
# -----------------------------------------------------------
# 🔹 5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
# -----------------------------------------------------------

print(df['city'].value_counts())   # ' Pune ', 'pune', 'Pune.' all look different!

city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: Int64


In [37]:
# -----------------------------------------------------------
# 🔹 5B. STANDARDISE THE STRINGS
# -----------------------------------------------------------

s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())        # now clean categories

city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


In [38]:
messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')

In [39]:
# strip + lower

print('Original:\n', messy)
messy = messy.str.strip().str.lower()
print('\nCleaned:\n', messy)

Original:
 0      London 
1       london
2       LONDON
3         N.Y.
4    new york 
5     New York
dtype: string

Cleaned:
 0      london
1      london
2      london
3        n.y.
4    new york
5    new york
dtype: string


In [40]:
# map 'n.y.' -> 'new york'  (after lowering)

messy = messy.replace({'n.y.': 'new york'})
print('After mapping n.y. to new york:\n', messy)

After mapping n.y. to new york:
 0      london
1      london
2      london
3    new york
4    new york
5    new york
dtype: string


In [41]:
# value_counts()

df_no_outlier = df[(df['spend'] >= low) & (df['spend'] <= high)].copy()
print(f"DataFrame shape before outlier removal: {df.shape}")
print(f"DataFrame shape after outlier removal: {df_no_outlier.shape}")
display(df_no_outlier)

DataFrame shape before outlier removal: (7, 7)
DataFrame shape after outlier removal: (6, 7)


,id,name,age,city,spend,date,spend_capped
0,1,Ana,30.0,pune,120.5,2024-01-05,120.5
1,2,Bo,25.0,pune,80.0,2024-01-06,80.0
2,3,Cy,29.5,delhi,200.2,2024-01-07,200.2
3,4,Di,41.0,delhi,120.5,2024-01-08,120.5
4,5,Eve,29.5,mumbai,150.0,2024-01-09,150.0
6,7,Gus,29.0,pune,110.0,2024-01-11,110.0


In [42]:
# After all the steps above, here's the cleaned frame
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean

Final shape: (7, 6)
Missing values: 0
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,120.5,2024-01-08
4,5,Eve,29.5,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11
